# 🎬 4DGaussians-Enhanced: Mask-Weighted Loss Training

This notebook provides a complete workflow for training 4D Gaussian Splatting with mask-weighted loss to improve dynamic scene reconstruction quality.

## Features
- ✅ SAM2 automatic mask generation
- ✅ Mask preview and validation
- ✅ Mask-weighted loss with foreground/background control
- ✅ Training presets (quick_test, standard, high_quality, fast_motion)
- ✅ Easy export and rendering

## Hardware Requirements
- Recommended: A100 (Colab Pro)
- Minimum: T4 (Free tier) - use quick_test preset

---

## 📦 Cell 2: Installation

Install 4DGaussians and SAM2 dependencies.

In [ ]:
# Clone repository
!git clone https://github.com/semhfe/4DGaussians-Enhanced.git
%cd 4DGaussians-Enhanced

# Install dependencies
!pip install -q -r requirements.txt
!pip install -q opencv-python

# Install submodules
!git submodule update --init --recursive
!pip install -q -e submodules/depth-diff-gaussian-rasterization
!pip install -q -e submodules/simple-knn

print("✅ Installation complete!")

## 📁 Cell 3: Mount Drive & Configure Paths

Mount Google Drive and set up paths to your data.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Configure paths
# EDIT THESE PATHS:
SOURCE_PATH = "/content/drive/MyDrive/4dgs_data/my_scene"  # Path to your scene data
OUTPUT_PATH = "/content/drive/MyDrive/4dgs_outputs/my_scene"  # Where to save results

# Verify paths exist
if not os.path.exists(SOURCE_PATH):
    print(f"⚠️ Warning: SOURCE_PATH does not exist: {SOURCE_PATH}")
    print("Please update SOURCE_PATH to point to your data.")
else:
    print(f"✅ Source path found: {SOURCE_PATH}")
    print(f"📁 Contents: {os.listdir(SOURCE_PATH)}")

# Create output directory
os.makedirs(OUTPUT_PATH, exist_ok=True)
print(f"✅ Output will be saved to: {OUTPUT_PATH}")

## 🎭 Cell 4: SAM2 Mask Generation

Generate masks automatically using SAM2.

In [ ]:
from utils.sam2_utils import generate_masks_for_scene

# SAM2 Configuration
# Detection prompt - objects to detect (comma-separated)
# Examples:
#   "person,human" - for human subjects
#   "person,human,bag,backpack" - person with accessories
#   "car,vehicle" - for vehicles
#   "dog,pet" - for pets
DETECTION_PROMPT = "person,human"  # @param {type:"string"}

# Model size - larger = better quality but slower
MODEL_SIZE = "large"  # @param ["small", "base", "large"]

# Confidence threshold - higher = stricter detection
CONFIDENCE_THRESHOLD = 0.5  # @param {type:"slider", min:0.1, max:0.9, step:0.05}

# Process every N frames - set to 1 for all frames, higher for faster processing
EVERY_N_FRAMES = 1  # @param {type:"integer"}

print("🎭 Generating masks with SAM2...")
print(f"  Prompt: {DETECTION_PROMPT}")
print(f"  Model: {MODEL_SIZE}")
print(f"  Threshold: {CONFIDENCE_THRESHOLD}")
print(f"  Every N frames: {EVERY_N_FRAMES}")

# Generate masks
stats = generate_masks_for_scene(
    source_path=SOURCE_PATH,
    mask_folder="masks",
    prompt=DETECTION_PROMPT,
    threshold=CONFIDENCE_THRESHOLD,
    every_n=EVERY_N_FRAMES,
    model_size=MODEL_SIZE
)

print("\n✅ Mask generation complete!")
print(f"📊 Stats: {stats}")

## 👀 Cell 5: Mask Preview & Validation

Interactively preview generated masks to verify quality.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import glob
from ipywidgets import interact, IntSlider, Dropdown

# Find all cameras
cam_folders = sorted(glob.glob(os.path.join(SOURCE_PATH, "cam*")))

def preview_mask(camera_idx=0, frame_idx=1):
    cam_folder = cam_folders[camera_idx]
    cam_name = os.path.basename(cam_folder)
    
    # Load image
    frame_path = os.path.join(cam_folder, f"frame_{str(frame_idx).zfill(5)}.jpg")
    if not os.path.exists(frame_path):
        frame_path = frame_path.replace(".jpg", ".png")
    
    # Load mask
    mask_path = os.path.join(cam_folder, "masks", f"mask_{str(frame_idx).zfill(5)}.png")
    
    if not os.path.exists(frame_path):
        print(f"Frame not found: {frame_path}")
        return
    
    image = np.array(Image.open(frame_path))
    
    if os.path.exists(mask_path):
        mask = np.array(Image.open(mask_path).convert('L'))
        # Create overlay
        overlay = image.copy()
        # Tint foreground green
        overlay[:,:,1] = np.where(mask > 128, np.minimum(overlay[:,:,1] + 50, 255), overlay[:,:,1])
    else:
        mask = np.ones((image.shape[0], image.shape[1]), dtype=np.uint8) * 255
        overlay = image
        print(f"⚠️ Mask not found: {mask_path}")
    
    # Display
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(image)
    axes[0].set_title(f"{cam_name} - Frame {frame_idx}\nOriginal")
    axes[0].axis('off')
    
    axes[1].imshow(mask, cmap='gray')
    axes[1].set_title("Mask\n(White=Foreground, Black=Background)")
    axes[1].axis('off')
    
    axes[2].imshow(overlay)
    axes[2].set_title("Overlay\n(Green tint = Foreground)")
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Show mask coverage
    fg_pixels = np.sum(mask > 128)
    total_pixels = mask.size
    fg_percent = 100 * fg_pixels / total_pixels
    print(f"📊 Foreground coverage: {fg_percent:.1f}%")

# Get frame count from first camera
first_cam_frames = glob.glob(os.path.join(cam_folders[0], "frame_*.jpg"))
num_frames = len(first_cam_frames)

# Interactive preview
interact(
    preview_mask,
    camera_idx=IntSlider(min=0, max=len(cam_folders)-1, step=1, value=0, description='Camera:'),
    frame_idx=IntSlider(min=1, max=num_frames, step=1, value=1, description='Frame:')
)

## 🎯 Cell 6: Decision Point

Choose your next step based on mask quality.

In [ ]:
print("📋 Mask Quality Check:")
print("")
print("✅ If masks look good:")
print("   → Set masks_ok = True and run Cell 7 (Training Configuration)")
print("")
print("⚠️ If masks need adjustment:")
print("   → Go back to Cell 4 and try:")
print("      • Different detection prompt")
print("      • Adjust confidence threshold")
print("      • Try different model size")
print("")
print("🛠️ If masks are inverted (foreground is black):")
print("   → Run: !python scripts/invert_masks.py --source_path {SOURCE_PATH} --inplace")
print("")
print("❌ If you want to skip masks:")
print("   → Set use_mask_loss = False in Cell 7")
print("")

# Set this flag when ready to proceed
masks_ok = True  # @param {type:"boolean"}

## ⚙️ Cell 7: Training Configuration

Configure training parameters using presets or custom values.

In [ ]:
# Training Presets
PRESETS = {
    "quick_test": {
        "iterations": 14000,
        "coarse_iterations": 2000,
        "w_fg": 1.0,
        "w_bg": 0.1,
        "description": "Fast test run (~30 min on A100, ~1h on T4)"
    },
    "standard": {
        "iterations": 30000,
        "coarse_iterations": 3000,
        "w_fg": 1.0,
        "w_bg": 0.1,
        "description": "Balanced quality/speed (~1.5 hours on A100)"
    },
    "high_quality": {
        "iterations": 60000,
        "coarse_iterations": 5000,
        "w_fg": 1.2,
        "w_bg": 0.05,
        "net_width": 128,
        "description": "Best quality (~3-4 hours on A100)"
    },
    "fast_motion": {
        "iterations": 45000,
        "coarse_iterations": 3000,
        "w_fg": 1.5,
        "w_bg": 0.1,
        "defor_depth": 2,
        "time_smoothness_weight": 0.005,
        "description": "Optimized for dancing/action scenes (~2 hours on A100)"
    }
}

# Select preset
PRESET = "standard"  # @param ["quick_test", "standard", "high_quality", "fast_motion"]

# Basic parameters (always visible)
USE_MASK_LOSS = True  # @param {type:"boolean"}
ITERATIONS = PRESETS[PRESET]["iterations"]  # @param {type:"integer"}
W_FG = PRESETS[PRESET]["w_fg"]  # @param {type:"number"}
W_BG = PRESETS[PRESET]["w_bg"]  # @param {type:"number"}

# Advanced parameters (toggle to show)
SHOW_ADVANCED = False  # @param {type:"boolean"}

if SHOW_ADVANCED:
    BATCH_SIZE = 1  # @param {type:"integer"}
    LAMBDA_DSSIM = 0.0  # @param {type:"number"}
    DENSIFY_UNTIL_ITER = 15000  # @param {type:"integer"}
    NET_WIDTH = PRESETS[PRESET].get("net_width", 64)  # @param {type:"integer"}
    DEFOR_DEPTH = PRESETS[PRESET].get("defor_depth", 1)  # @param {type:"integer"}
    TIME_SMOOTHNESS_WEIGHT = PRESETS[PRESET].get("time_smoothness_weight", 0.01)  # @param {type:"number"}
else:
    BATCH_SIZE = 1
    LAMBDA_DSSIM = 0.0
    DENSIFY_UNTIL_ITER = 15000
    NET_WIDTH = PRESETS[PRESET].get("net_width", 64)
    DEFOR_DEPTH = PRESETS[PRESET].get("defor_depth", 1)
    TIME_SMOOTHNESS_WEIGHT = PRESETS[PRESET].get("time_smoothness_weight", 0.01)

print(f"🎯 Selected preset: {PRESET}")
print(f"   {PRESETS[PRESET]['description']}")
print("")
print("📊 Configuration:")
print(f"   Iterations: {ITERATIONS}")
print(f"   Mask-weighted loss: {USE_MASK_LOSS}")
if USE_MASK_LOSS:
    print(f"   Foreground weight (w_fg): {W_FG}")
    print(f"   Background weight (w_bg): {W_BG}")
print("")
print("✅ Configuration ready! Run Cell 8 to start training.")

## 🚀 Cell 8: Start Training

Run the training process.

In [ ]:
import time

# Build command
cmd = f"""python train.py \
    --source_path {SOURCE_PATH} \
    --model_path {OUTPUT_PATH} \
    --iterations {ITERATIONS} \
    --coarse_iterations {PRESETS[PRESET]['coarse_iterations']} \
    --batch_size {BATCH_SIZE} \
    --lambda_dssim {LAMBDA_DSSIM} \
    --densify_until_iter {DENSIFY_UNTIL_ITER} \
    --net_width {NET_WIDTH} \
    --defor_depth {DEFOR_DEPTH} \
    --time_smoothness_weight {TIME_SMOOTHNESS_WEIGHT}"""

if USE_MASK_LOSS:
    cmd += f" --use_mask_loss --w_fg {W_FG} --w_bg {W_BG}"

print("🚀 Starting training...")
print(f"📝 Command: {cmd}")
print("")

start_time = time.time()

# Run training
!{cmd}

elapsed = time.time() - start_time
print("")
print(f"✅ Training complete! Time: {elapsed/3600:.2f} hours")
print(f"📁 Model saved to: {OUTPUT_PATH}")

## 🎥 Cell 9: Render Results

Render the trained model to video.

In [ ]:
# Render video
render_cmd = f"""python render.py \
    --source_path {SOURCE_PATH} \
    --model_path {OUTPUT_PATH} \
    --iteration {ITERATIONS}"""

print("🎥 Rendering video...")
!{render_cmd}

# Find rendered video
import glob
video_files = glob.glob(os.path.join(OUTPUT_PATH, "**/*.mp4"), recursive=True)

if video_files:
    print("")
    print("✅ Rendering complete!")
    print(f"📹 Video saved to: {video_files[0]}")
    
    # Display video
    from IPython.display import Video
    display(Video(video_files[0], width=800))
else:
    print("⚠️ No video files found. Check output directory.")

## 💾 Cell 10: Export PLY Sequence

Export per-frame 3D Gaussian point clouds.

In [ ]:
# Export per-frame PLY files
export_cmd = f"""python export_perframe_3DGS.py \
    --source_path {SOURCE_PATH} \
    --model_path {OUTPUT_PATH} \
    --iteration {ITERATIONS}"""

print("💾 Exporting PLY sequence...")
!{export_cmd}

ply_dir = os.path.join(OUTPUT_PATH, "per_frame_ply")
if os.path.exists(ply_dir):
    ply_files = glob.glob(os.path.join(ply_dir, "*.ply"))
    print("")
    print(f"✅ Export complete! {len(ply_files)} PLY files saved.")
    print(f"📁 Location: {ply_dir}")
else:
    print("⚠️ Export directory not found. Check command output.")

print("")
print("🎉 All done! Your 4D Gaussian model is ready.")